In [ ]:
# TF/IDF

# source (https://www.geeksforgeeks.org/machine-learning/understanding-tf-idf-term-frequency-inverse-document-frequency/)
import math 

def tf(term: str, document: str, counted_plural: bool) -> float:
    """
    counted_plural: if True, then count the plural form of the term as well
    """
    # calculate term frequency
    doc_length = len(document.split())

    # hashmap to store the frequency of each term in the document
    map = {}
    for word in document.split():
        if word in map:
            map[word] += 1
        else:
            map[word] = 1

    if counted_plural:
        tf = (map.get(term, 0) + map.get(term + "s", 0)) / doc_length
    else:
        tf = map.get(term, 0) / doc_length

    return tf

def idf(term: str, documents: list, counted_plural: bool) -> float:
    # n docs
    N = len(documents)

    # count the number of documents that contain the term
    count = 0
    for document in documents:
        if counted_plural:
            if term in document.split() or term + "s" in document.split():
                count += 1
        else:
            if term in document.split():
                count += 1

    idf = 0
    if count > 0:
        idf = math.log(N / count, 10)

    return idf

def tf_idf(term: str, document: str, documents: list, counted_plural: bool, is_lower: bool) -> float:
    if is_lower:
        term = term.lower()
        document = document.lower()
        documents = [doc.lower() for doc in documents]

    return tf(term, document, counted_plural) * idf(term, documents, counted_plural)

doc_1 = "the cats sat on the mat"
doc_2 = "the dog sat on the log"
doc_3 = "the cat chased the dog"

doc_lower_test = "THE CAT CHASED THE DOG"
print(doc_lower_test.lower())

print(tf("cat", doc_1, True))
print(idf("cat", [doc_1, doc_2, doc_3], True))
print(tf_idf("cat", doc_1, [doc_1, doc_2, doc_3], counted_plural=True, is_lower=False))

the cat chased the dog
0.16666666666666666
0.17609125905568124
0.029348543175946873


In [20]:
#BM25
# source https://www.geeksforgeeks.org/nlp/what-is-bm25-best-matching-25-algorithm/

def tf_bm25(term: str, document: str, counted_plural: bool, k1: float, b: float, avg_doc_length: float) -> float:
    # bigger k1 -> smaller tf -> term less important
    # b: scaling factor for doc length compared to avg doc length
    # -> bigger b -> Length Normalization more important 
    doc_length = len(document.split())
    map = {}
    freq_t_d = 0

    for word in document.split():
        if word in map:
            map[word] += 1
        else:
            map[word] = 1

    if counted_plural:
        freq_t_d = (map.get(term, 0) + map.get(term + "s", 0))
    else:
        freq_t_d = map.get(term, 0)

    tf = freq_t_d / (freq_t_d + k1 * (1 - b + b * doc_length / avg_doc_length))
    
    return tf

def idf_bm25(term: str, documents: list, counted_plural: bool) -> float:
    N = len(documents)
    count = 0

    for document in documents:
        if counted_plural:
            if term in document.split() or term + "s" in document.split():
                count += 1
        else:
            if term in document.split():
                count += 1

    idf = 0
    if count > 0:
        idf = math.log((N - count + 0.5) / (count + 0.5), 10)

    return idf

def bm25(query: str, document: str, documents: list, counted_plural: bool, k1: float, b: float) -> float:
    avg_doc_length = sum(len(doc.split()) for doc in documents) / len(documents)

    score = 0
    for term in query.split():
        score += tf_bm25(term, document, counted_plural, k1, b, avg_doc_length) * idf_bm25(term, documents, counted_plural)

    return score

# to compare https://github.com/ev2900/BM25_Search_Example
doc_bm_1 = "Hello there good man!"
doc_bm_2 = "It is quite windy in London"
doc_bm_3 = "How is the weather today?"

query = "windy London"

for doc in [doc_bm_1, doc_bm_2, doc_bm_3]:
    print(bm25(query, doc, [doc_bm_1, doc_bm_2, doc_bm_3], False, k1=1.5, b=0.75))

0.0
0.16282477036062853
0.0
